# 01 — API Calls & Requests

`🔴 Advanced`

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaIslamFahad/python-for-ai-engineers/blob/main/07_AI_Engineering/01_api_calls_and_requests.ipynb)

## 📌 What is it?
The `requests` library is Python's standard for making HTTP calls. Understanding REST APIs, authentication, sessions, and error handling is foundational for all LLM work.

## 🧠 Why AI Engineers need this
Every LLM call is an HTTP POST request. SDKs abstract this, but knowing the underlying mechanics helps debug auth errors, rate limits, timeouts, and streaming.

In [ ]:
import requests
import json
import os

# ── BASIC GET REQUEST ──
response = requests.get("https://httpbin.org/get", timeout=10)

print(f"Status Code: {response.status_code}")   # 200
print(f"Content-Type: {response.headers['content-type']}")
print(f"Response JSON keys: {list(response.json().keys())}")

In [ ]:
import requests
import json
import time

# ── MAKING LLM API CALLS MANUALLY ──
def call_openai(prompt: str, api_key: str, model: str = "gpt-4o") -> dict:
    """Make a raw OpenAI API call with requests."""
    url = "https://api.openai.com/v1/chat/completions"
    
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
    }
    
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7,
        "max_tokens": 500
    }
    
    response = requests.post(url, headers=headers, json=payload, timeout=30)
    response.raise_for_status()   # raise on 4xx/5xx
    return response.json()

# Usage:
# api_key = os.environ["OPENAI_API_KEY"]  # never hardcode!
# result = call_openai("What is RAG?", api_key)
# print(result["choices"][0]["message"]["content"])

print("API call function defined. Set OPENAI_API_KEY env var to use.")

In [ ]:
import requests
import time

# ── RETRY WITH EXPONENTIAL BACKOFF ──
def robust_api_call(
    url: str,
    headers: dict,
    payload: dict,
    max_retries: int = 3,
    base_delay: float = 1.0
) -> dict:
    """Make an API call with automatic retry on failure."""
    for attempt in range(max_retries):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=30)
            
            if response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", base_delay * (2**attempt)))
                print(f"Rate limited. Waiting {retry_after}s...")
                time.sleep(retry_after)
                continue
            
            if response.status_code == 500:
                wait = base_delay * (2 ** attempt)
                print(f"Server error. Waiting {wait}s... (attempt {attempt+1})")
                time.sleep(wait)
                continue
            
            response.raise_for_status()
            return response.json()
        
        except requests.Timeout:
            wait = base_delay * (2 ** attempt)
            print(f"Timeout on attempt {attempt+1}. Waiting {wait}s...")
            time.sleep(wait)
        
        except requests.ConnectionError as e:
            print(f"Connection error: {e}")
            raise
    
    raise RuntimeError(f"Failed after {max_retries} retries")

print("Robust API call function ready.")
print("Handles: 429 rate limit, 500 server error, timeout, connection error")

In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import json

# ── SESSIONS (reuse connections, share config) ──
def create_api_session(api_key: str, max_retries: int = 3) -> requests.Session:
    """Create a configured session for repeated API calls."""
    session = requests.Session()
    
    # Set default headers
    session.headers.update({
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "User-Agent": "MyAIApp/1.0"
    })
    
    # Configure automatic retries
    retry_strategy = Retry(
        total=max_retries,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["POST", "GET"]
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    
    return session

# Usage
# session = create_api_session(os.environ["OPENAI_API_KEY"])
# for prompt in prompts:
#     r = session.post(url, json={"messages":[{"role":"user","content":prompt}]})
print("Session factory created. Sessions reuse TCP connections — faster for multiple calls!")

In [ ]:
import asyncio
import json

# ── ASYNC HTTP with aiohttp (parallel LLM calls) ──
async_example = '''
import aiohttp
import asyncio

async def call_llm_async(session, prompt: str) -> str:
    url = "https://api.openai.com/v1/chat/completions"
    payload = {
        "model": "gpt-4o",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 200
    }
    async with session.post(url, json=payload) as resp:
        data = await resp.json()
        return data["choices"][0]["message"]["content"]

async def batch_llm_calls(prompts: list[str], api_key: str) -> list[str]:
    headers = {"Authorization": f"Bearer {api_key}"}
    async with aiohttp.ClientSession(headers=headers) as session:
        tasks = [call_llm_async(session, p) for p in prompts]
        return await asyncio.gather(*tasks)   # all in parallel!

# prompts = ["What is RAG?", "What is LoRA?", "What is RLHF?"]
# results = asyncio.run(batch_llm_calls(prompts, api_key))
# 3 calls in parallel ≈ same time as 1 sequential call!
'''
print(async_example)
print("\nKey insight: asyncio.gather() runs all API calls concurrently!")
print("For 10 calls: sequential = 10x latency, async = ~1x latency")

## ⚠️ Common Mistakes
```python
# ❌ Hardcoding API keys
api_key = "sk-abc123"   # NEVER commit to git!

# ✅ Load from environment
import os
api_key = os.environ["OPENAI_API_KEY"]

# ❌ No timeout — hangs forever
requests.post(url, json=payload)

# ✅ Always set timeout
requests.post(url, json=payload, timeout=30)

# ❌ Not checking status code
response = requests.post(url, ...)
data = response.json()   # might be an error response!

# ✅
response.raise_for_status()   # raises on 4xx/5xx
data = response.json()
```

## 🔗 What's Next?
[02 — Working with LLMs →](02_working_with_llms.ipynb)